# 08 · Design Principles & Patterns in Python (SOLID, GoF, Pythonic Idioms)

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 70 min

### 🎯 What you will learn
- **SOLID** principles with Python examples
- Other key principles: DRY, KISS, YAGNI, composition over inheritance, "explicit is better than implicit"
- The most useful design patterns, written the **Pythonic** way (often with functions, not classes):
  - Creational: **Factory**, **Builder**, **Singleton** (and why modules suffice)
  - Structural: **Adapter**, **Decorator**, **Facade**, **Proxy**
  - Behavioural: **Strategy**, **Observer**, **Command**, **Iterator**, **Template Method**, **State**
- **Dependency injection**, the **Repository** pattern, layered architecture
- Anti-patterns to avoid

> 💡 Python's first-class functions, duck typing, modules and decorators make many classic "Gang of Four" patterns **much simpler** than in Java/C++.

## 1. SOLID

### S: Single Responsibility
A class/function should have **one reason to change**.

In [1]:
# ❌ one class does everything
class ReportBad:
    def __init__(self, rows): self.rows = rows
    def total(self): return sum(r["amount"] for r in self.rows)
    def to_csv(self): return "\n".join(f'{r["item"]},{r["amount"]}' for r in self.rows)
    def email(self, to): print(f"emailing {to}...")          # formatting + delivery mixed with logic

# ✅ split responsibilities
class Report:
    def __init__(self, rows): self.rows = rows
    def total(self): return sum(r["amount"] for r in self.rows)

def to_csv(report: Report) -> str:
    return "\n".join(f'{r["item"]},{r["amount"]}' for r in report.rows)

class EmailSender:
    def send(self, to: str, body: str) -> None:
        print(f"[email to {to}] {body[:30]}...")

r = Report([{"item": "pen", "amount": 10}, {"item": "book", "amount": 250}])
EmailSender().send("boss@example.com", to_csv(r))
print(r.total())

[email to boss@example.com] pen,10
book,250...
260


### O: Open/Closed
Open for **extension**, closed for **modification**: add behaviour without editing existing code.

In [2]:
# ❌ every new format means editing this function
def export_bad(data, fmt):
    if fmt == "csv": ...
    elif fmt == "json": ...
    # elif fmt == "xml": ...  ← edit again

# ✅ register new exporters without touching existing code
import json
EXPORTERS = {}
def exporter(fmt):
    def register(func):
        EXPORTERS[fmt] = func
        return func
    return register

@exporter("json")
def to_json(data): return json.dumps(data)

@exporter("csv")
def to_csv_rows(data): return "\n".join(",".join(map(str, row.values())) for row in data)

def export(data, fmt):
    return EXPORTERS[fmt](data)

print(export([{"a": 1, "b": 2}], "json"), "|", export([{"a": 1, "b": 2}], "csv"))

[{"a": 1, "b": 2}] | 1,2


### L: Liskov Substitution
Subclasses must be usable **wherever the base class is expected**, without surprises.

In [3]:
class Rectangle:
    def __init__(self, w, h): self.w, self.h = w, h
    def set_width(self, w): self.w = w
    def area(self): return self.w * self.h

class Square(Rectangle):                    # ❌ classic LSP violation
    def __init__(self, side): super().__init__(side, side)
    def set_width(self, w): self.w = self.h = w      # changes height too!

def stretch(rect: Rectangle):
    rect.set_width(10)
    return rect.area()                      # caller expects 10 * h

print(stretch(Rectangle(2, 3)), stretch(Square(3)))    # 30 vs 100: surprising for callers

30 100


✅ Fix: don't model `Square` as a subtype of a *mutable* `Rectangle`; use a common `Shape` protocol with `area()`, or immutable value objects.

### I: Interface Segregation
Many small, focused interfaces beat one big one. In Python: small **Protocols**.

In [4]:
from typing import Protocol

class Readable(Protocol):
    def read(self) -> str: ...
class Writable(Protocol):
    def write(self, data: str) -> None: ...

class Clipboard:                      # implements only what it needs
    def __init__(self): self._data = ""
    def read(self) -> str: return self._data
    def write(self, data: str) -> None: self._data = data

def copy_text(src: Readable, dst: Writable) -> None:
    dst.write(src.read())

a, b = Clipboard(), Clipboard()
a.write("hello")
copy_text(a, b)
print(b.read())

hello


### D: Dependency Inversion
High-level code should depend on **abstractions**, not concrete details, which makes swapping and testing easy.

In [5]:
class Notifier(Protocol):
    def send(self, to: str, message: str) -> None: ...

class SMSNotifier:
    def send(self, to, message): print(f"SMS to {to}: {message}")

class FakeNotifier:                         # for tests
    def __init__(self): self.sent = []
    def send(self, to, message): self.sent.append((to, message))

class OrderService:
    def __init__(self, notifier: Notifier):        # dependency INJECTED, not created inside
        self.notifier = notifier
    def place(self, customer, item):
        self.notifier.send(customer, f"Order for {item} confirmed")

OrderService(SMSNotifier()).place("+91-98765", "laptop")
fake = FakeNotifier()
OrderService(fake).place("asha", "pen")
print(fake.sent)

SMS to +91-98765: Order for laptop confirmed
[('asha', 'Order for pen confirmed')]


## 2. Other principles
- **DRY** (Don't Repeat Yourself): but don't abstract too early; "rule of three"
- **KISS** (Keep It Simple) and **YAGNI** (You Aren't Gonna Need It)
- **Composition over inheritance** (Module 06 · 02)
- **Law of Demeter**: talk to friends, not friends of friends (`order.customer.address.city` chains are a smell)
- **Fail fast**: validate inputs at boundaries (Pydantic, Module 15)
- **The Zen of Python**: `import this`

## 3. Creational patterns

### Factory: functions or classmethods

In [6]:
from dataclasses import dataclass

@dataclass
class Connection:
    url: str
    pool_size: int = 5

    @classmethod
    def for_testing(cls):                           # factory classmethod
        return cls("sqlite:///:memory:", pool_size=1)

def make_connection(env: str) -> Connection:         # factory function
    return {"prod": Connection("postgresql://prod", 20),
            "dev": Connection("postgresql://localhost")}.get(env) or Connection.for_testing()

print(make_connection("prod"), make_connection("ci"))

Connection(url='postgresql://prod', pool_size=20) Connection(url='sqlite:///:memory:', pool_size=1)


### Builder: fluent construction of complex objects

In [7]:
from typing import Self

class QueryBuilder:
    def __init__(self, table: str):
        self._table, self._where, self._order, self._limit = table, [], None, None
    def where(self, cond: str) -> Self:
        self._where.append(cond); return self
    def order_by(self, col: str) -> Self:
        self._order = col; return self
    def limit(self, n: int) -> Self:
        self._limit = n; return self
    def build(self) -> str:
        sql = f"SELECT * FROM {self._table}"
        if self._where: sql += " WHERE " + " AND ".join(self._where)
        if self._order: sql += f" ORDER BY {self._order}"
        if self._limit: sql += f" LIMIT {self._limit}"
        return sql

print(QueryBuilder("users").where("age > 18").where("active").order_by("name").limit(10).build())

SELECT * FROM users WHERE age > 18 AND active ORDER BY name LIMIT 10


> 💡 In Python, keyword arguments with defaults (and dataclasses) remove the need for many builders.

### Singleton: just use a module

```python
# settings.py
DEBUG = False
DATABASE_URL = "postgresql://..."
# anywhere: from myapp import settings  →  the same module object everywhere
```
Modules are imported once and cached in `sys.modules`. Use `functools.cache` on a factory function for lazy singletons (`@cache def get_settings(): ...`, the pattern FastAPI recommends).

## 4. Structural patterns

### Adapter: make an incompatible interface fit

In [8]:
class LegacyPrinter:
    def print_text_upper(self, s): print("LEGACY:", s.upper())

class PrinterAdapter:                      # our code expects .write(str)
    def __init__(self, legacy): self.legacy = legacy
    def write(self, data: str) -> None: self.legacy.print_text_upper(data)

copy_text(a, PrinterAdapter(LegacyPrinter()))

LEGACY: HELLO


### Decorator pattern: add behaviour by wrapping (Python's `@` syntax is one form)

In [9]:
class DataSource(Protocol):
    def read(self) -> str: ...

class FileSource:
    def read(self): return "secret data"

class EncryptedSource:                      # wraps any DataSource
    def __init__(self, inner: DataSource): self.inner = inner
    def read(self): return self.inner.read()[::-1]       # pretend encryption

class LoggingSource:
    def __init__(self, inner: DataSource): self.inner = inner
    def read(self):
        data = self.inner.read()
        print(f"read {len(data)} chars")
        return data

print(LoggingSource(EncryptedSource(FileSource())).read())

read 11 chars
atad terces


### Facade: a simple front for a complex subsystem

In [10]:
class Inventory:
    def reserve(self, item): return f"reserved {item}"
class Payments:
    def charge(self, amount): return f"charged ₹{amount}"
class Shipping:
    def ship(self, item): return f"shipped {item}"

class Checkout:                             # one call hides three subsystems
    def __init__(self): self.inv, self.pay, self.ship = Inventory(), Payments(), Shipping()
    def buy(self, item, price):
        return [self.inv.reserve(item), self.pay.charge(price), self.ship.ship(item)]

print(Checkout().buy("laptop", 54999))

['reserved laptop', 'charged ₹54999', 'shipped laptop']


**Proxy**: an object standing in for another (lazy loading, access control, caching, remote calls). See `LazyProxy` in lesson 01.

## 5. Behavioural patterns

### Strategy: pass the algorithm as a function

In [11]:
from collections.abc import Callable

def no_discount(total): return total
def festive(total): return total * 0.8
def bulk(total): return total - 500 if total > 5000 else total

def checkout(total: float, pricing: Callable[[float], float] = no_discount) -> float:
    return pricing(total)

for strategy in (no_discount, festive, bulk):
    print(f"{strategy.__name__:<12} {checkout(6000, strategy)}")

no_discount  6000
festive      4800.0
bulk         5500


In Java you'd create a `DiscountStrategy` interface and three classes; in Python **functions are strategies** (`sorted(key=...)` is the strategy pattern too).

### Observer: publish/subscribe

In [12]:
from collections import defaultdict

class EventBus:
    def __init__(self):
        self._subscribers = defaultdict(list)
    def subscribe(self, event: str, handler: Callable) -> None:
        self._subscribers[event].append(handler)
    def publish(self, event: str, **data) -> None:
        for handler in self._subscribers[event]:
            handler(**data)

bus = EventBus()
bus.subscribe("user_signed_up", lambda email: print(f"send welcome email to {email}"))
bus.subscribe("user_signed_up", lambda email: print(f"add {email} to CRM"))
bus.publish("user_signed_up", email="asha@example.com")

send welcome email to asha@example.com
add asha@example.com to CRM


### Command: actions as objects (undo/redo, queues)

In [13]:
@dataclass
class AppendText:
    doc: list
    text: str
    def execute(self): self.doc.append(self.text)
    def undo(self): self.doc.pop()

doc, history = [], []
for cmd in (AppendText(doc, "Hello"), AppendText(doc, "World")):
    cmd.execute(); history.append(cmd)
print(doc)
history.pop().undo()
print(doc)

['Hello', 'World']
['Hello']


### Iterator: generators (Module 09 · 02). Template Method: a base class defines the skeleton, subclasses fill in steps

In [14]:
from abc import ABC, abstractmethod

class DataPipeline(ABC):
    def run(self):                     # the template: fixed order of steps
        data = self.extract()
        data = self.transform(data)
        return self.load(data)
    @abstractmethod
    def extract(self): ...
    def transform(self, data): return data          # optional hook
    @abstractmethod
    def load(self, data): ...

class NumbersPipeline(DataPipeline):
    def extract(self): return ["3", "1", "2"]
    def transform(self, data): return sorted(map(int, data))
    def load(self, data): return f"loaded {data}"

print(NumbersPipeline().run())

loaded [1, 2, 3]


### State: behaviour changes with internal state

In [15]:
class OrderState(ABC):
    @abstractmethod
    def next(self, order): ...

class Pending(OrderState):
    def next(self, order): order.state = Shipped(); return "shipped"
class Shipped(OrderState):
    def next(self, order): order.state = Delivered(); return "delivered"
class Delivered(OrderState):
    def next(self, order): return "already delivered"

class Order:
    def __init__(self): self.state = Pending()
    def advance(self): return self.state.next(self)

o = Order()
print([o.advance() for _ in range(4)])

['shipped', 'delivered', 'already delivered', 'already delivered']


For simple cases, an `Enum` + `match` (Module 04 · 02) is often clearer than state classes.

## 6. Repository pattern & layered architecture

Keep **business logic** independent of **storage** and **frameworks**:

```
 presentation (CLI / FastAPI routes)      ← thin
        │
 service layer (business rules)           ← depends on abstractions
        │
 repository interface (Protocol)
        │
 infrastructure (SQLAlchemy / files / HTTP APIs)
```

In [16]:
class UserRepository(Protocol):
    def get(self, user_id: int) -> dict | None: ...
    def add(self, user: dict) -> int: ...

class InMemoryUserRepository:
    def __init__(self): self._rows = {}
    def get(self, user_id): return self._rows.get(user_id)
    def add(self, user):
        uid = len(self._rows) + 1
        self._rows[uid] = {**user, "id": uid}
        return uid

class UserService:
    def __init__(self, repo: UserRepository): self.repo = repo
    def register(self, name: str, email: str) -> int:
        if "@" not in email:
            raise ValueError("invalid email")
        return self.repo.add({"name": name, "email": email})

service = UserService(InMemoryUserRepository())      # swap for a SQLRepository in production
uid = service.register("Asha", "asha@example.com")
print(service.repo.get(uid))

{'name': 'Asha', 'email': 'asha@example.com', 'id': 1}


## 7. Anti-patterns to avoid
| Anti-pattern | Instead |
|---|---|
| **God object**: one class that knows everything | split responsibilities |
| Deep inheritance hierarchies | composition, protocols |
| Java-style getters/setters everywhere | plain attributes → `@property` when needed |
| Classes with one method + `__init__` | a function (or `functools.partial`) |
| Global mutable state | dependency injection, explicit parameters |
| Catch-all `except Exception: pass` | handle specific errors |
| Premature abstraction / patterns "just because" | YAGNI; refactor when the need appears |
| Circular imports between modules | restructure layers; import inside functions as a last resort |

## 📝 Summary
- SOLID: single responsibility, open/closed, Liskov, interface segregation (small Protocols), dependency inversion (injection).
- Many GoF patterns collapse to **functions, decorators, modules and protocols** in Python.
- Repository + service layers keep business logic testable and framework-independent.

## 🏋️ Practice
1. Refactor a function with a long `if/elif` on a `payment_method` string into the Strategy pattern using a dict of functions.
2. Implement an `EventBus` with `unsubscribe` and wildcard (`"*"`) subscriptions.
3. Add undo/redo (two stacks) to the Command example.
4. Write a `FileUserRepository` (JSON file) with the same interface as `InMemoryUserRepository`, and run `UserService` with it unchanged.